# Final Project - Realized Volatility Timing

Standalone version using only local modules from `volatility_project/src`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.strategy.strategies import SHORT_1M_STRADDLE
from src.strategy.option_trade import DeltaHedgedOptionTrade
from src.data.option_loader import OptionLoader, extract_spot_from_options
from src.backtest.backtester import BacktesterBidAskFromData
from src.models.heston_kalman import HestonKalmanEstimator
from src.signal.vol_signal import build_vol_signal, compute_strategy_iv_reference, rolling_realized_vol_benchmark, map_signal_to_trade_entries
from src.allocation.dynamic_allocation import tanh_allocation

pd.options.display.float_format = '{:.4f}'.format

## Baseline positions

In [ ]:
start_date = pd.Timestamp('2020-01-02').to_pydatetime()
end_date = pd.Timestamp('2022-12-30').to_pydatetime()
ticker = 'SPY'

positions = DeltaHedgedOptionTrade.generate_trades(
    start_date=start_date,
    end_date=end_date,
    tickers=ticker,
    legs=SHORT_1M_STRADDLE,
)
positions.head()

## Realized volatility estimate

In [ ]:
option_data = OptionLoader.load_data(start_date, end_date, process_kwargs={'ticker': ticker})
spot = extract_spot_from_options(option_data).set_index('date')['spot']
estimator = HestonKalmanEstimator(kappa=3.0, process_noise=0.05, observation_noise=5e-7, auto_calibrate=True)
sigma_hat = estimator.fit_transform(spot)
sigma_hat = estimator.add_horizon_forecast(sigma_hat, horizon_days=28, prefix='forecast')
rolling_vol = rolling_realized_vol_benchmark(spot, window=21)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sigma_hat['date'], y=sigma_hat['forecast_sigma_hat'], mode='lines', name='Kalman 1M forecast sigma_hat'))
fig.add_trace(go.Scatter(x=rolling_vol['date'], y=rolling_vol['rolling_realized_vol'], mode='lines', name='Rolling realized vol (21d)'))
fig.update_layout(template='plotly_white', height=420, width=950, title='Kalman vs rolling realized volatility')
fig.update_xaxes(title='Date')
fig.update_yaxes(title='Annualized volatility')
fig.show()

## Signal and dynamic allocation

In [ ]:
iv_reference = compute_strategy_iv_reference(positions, option_data)
signal_definition = 'iv_minus_sigma'
scale_grid = [3.0, 5.0, 10.0]
signal = build_vol_signal(
    iv_reference,
    sigma_hat[['date', 'forecast_sigma_hat']],
    signal_definition=signal_definition,
    winsorize_quantiles=(0.01, 0.99),
)
allocation_grid = pd.DataFrame({'date': signal['date'], 'vol_signal': signal['vol_signal']})
for scale in scale_grid:
    allocation_grid[f'allocation_scale_{int(scale)}'] = tanh_allocation(
        signal['vol_signal'],
        scale=scale,
        leverage_cap=1.75,
        base=1.0,
        floor=0.25,
        increasing=True,
    )
chosen_scale = 5.0
signal['allocation_multiplier'] = allocation_grid[f'allocation_scale_{int(chosen_scale)}']
signal.tail()

In [ ]:
sensitivity_rows = []
for scale in scale_grid:
    temp_signal = signal[['date', 'vol_signal']].copy()
    temp_signal['allocation_multiplier'] = allocation_grid[f'allocation_scale_{int(scale)}']
    temp_positions = map_signal_to_trade_entries(positions, temp_signal[['date', 'allocation_multiplier']], lag_business_days=1)
    temp_positions['weight'] = temp_positions['weight'] * temp_positions['allocation_multiplier']
    temp_backtest = BacktesterBidAskFromData(temp_positions[['date', 'option_id', 'entry_date', 'leg_name', 'weight', 'ticker']]).compute_backtest()
    temp_returns = temp_backtest.nav['NAV'].pct_change().dropna()
    sensitivity_rows.append({
        'scale': scale,
        'annualized_return': temp_returns.mean() * 252,
        'annualized_volatility': temp_returns.std() * (252 ** 0.5),
        'sharpe': (temp_returns.mean() * 252) / (temp_returns.std() * (252 ** 0.5)) if temp_returns.std() > 0 else float('nan'),
    })
allocation_sensitivity = pd.DataFrame(sensitivity_rows)
allocation_sensitivity

## Baseline vs dynamic backtest

In [ ]:
baseline = BacktesterBidAskFromData(positions).compute_backtest()
dynamic_positions = map_signal_to_trade_entries(positions, signal[['date', 'allocation_multiplier']], lag_business_days=1)
dynamic_positions['weight'] = dynamic_positions['weight'] * dynamic_positions['allocation_multiplier']
dynamic = BacktesterBidAskFromData(dynamic_positions[['date', 'option_id', 'entry_date', 'leg_name', 'weight', 'ticker']]).compute_backtest()

nav_compare = baseline.nav.rename(columns={'NAV': 'baseline'}).join(dynamic.nav.rename(columns={'NAV': 'dynamic'}), how='outer').ffill()
fig = go.Figure()
fig.add_trace(go.Scatter(x=nav_compare.index, y=nav_compare['baseline'], mode='lines', name='Baseline'))
fig.add_trace(go.Scatter(x=nav_compare.index, y=nav_compare['dynamic'], mode='lines', name='Dynamic'))
fig.update_layout(template='plotly_white', height=420, width=950, title='NAV comparison')
fig.update_xaxes(title='Date')
fig.update_yaxes(title='NAV')
fig.show()

## Performance metrics

In [ ]:
def compute_performance_metrics(nav_df):
    returns = nav_df['NAV'].pct_change().dropna()
    annualized_return = returns.mean() * 252
    annualized_volatility = returns.std() * (252 ** 0.5)
    sharpe = annualized_return / annualized_volatility if annualized_volatility != 0 else float('nan')
    cumulative_nav = (1 + returns).cumprod()
    drawdown = cumulative_nav / cumulative_nav.cummax() - 1
    max_drawdown = drawdown.min()
    calmar = annualized_return / abs(max_drawdown) if max_drawdown != 0 else float('inf')
    return pd.Series({
        'annualized_return': annualized_return,
        'annualized_volatility': annualized_volatility,
        'sharpe': sharpe,
        'max_drawdown': max_drawdown,
        'calmar': calmar,
    })

metrics_table = pd.concat(
    {
        'baseline': compute_performance_metrics(baseline.nav),
        'dynamic': compute_performance_metrics(dynamic.nav),
    },
    axis=1,
)
metrics_table

## Signal distribution

In [ ]:
hist = go.Figure()
hist.add_trace(go.Histogram(x=signal['vol_signal'].dropna(), nbinsx=40, name='vol_signal', marker_color='steelblue', opacity=0.85))
hist.add_vline(x=float(signal['vol_signal'].mean()), line_dash='dash', line_color='black', annotation_text='mean')
hist.update_layout(template='plotly_white', height=420, width=950, title='Distribution of vol timing signal', bargap=0.05)
hist.update_xaxes(title='vol_signal')
hist.update_yaxes(title='Count')
hist.show()

## Signal vs allocation

In [ ]:
fig = go.Figure()
for scale in scale_grid:
    fig.add_trace(go.Scatter(
        x=signal['vol_signal'],
        y=allocation_grid[f'allocation_scale_{int(scale)}'],
        mode='markers',
        name=f'scale={int(scale)}',
        marker=dict(size=7, opacity=0.45),
    ))
fig.update_layout(template='plotly_white', height=420, width=900, title='Signal vs allocation (IV - forecast spread)')
fig.update_xaxes(title='vol_signal')
fig.update_yaxes(title='allocation_multiplier')
fig.show()

## Allocation through time

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=signal['date'], y=signal['allocation_multiplier'], mode='lines', name='Allocation', line=dict(color='darkorange')))
fig.update_layout(template='plotly_white', height=420, width=950, title='Allocation multiplier through time (chosen scale=5, higher when IV exceeds forecast)')
fig.update_xaxes(title='Date')
fig.update_yaxes(title='allocation_multiplier')
fig.show()

## PnL comparison

In [ ]:
pnl_compare = baseline.pnl[['pnl']].rename(columns={'pnl': 'baseline'}).join(dynamic.pnl[['pnl']].rename(columns={'pnl': 'dynamic'}), how='outer').fillna(0.0)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=('Cumulative PnL comparison', 'Daily PnL comparison'))
fig.add_trace(go.Scatter(x=pnl_compare.index, y=pnl_compare['baseline'].cumsum(), mode='lines', name='Baseline cumulative'), row=1, col=1)
fig.add_trace(go.Scatter(x=pnl_compare.index, y=pnl_compare['dynamic'].cumsum(), mode='lines', name='Dynamic cumulative'), row=1, col=1)
fig.add_trace(go.Scatter(x=pnl_compare.index, y=pnl_compare['baseline'], mode='lines', name='Baseline daily', line=dict(width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=pnl_compare.index, y=pnl_compare['dynamic'], mode='lines', name='Dynamic daily', line=dict(width=1)), row=2, col=1)
fig.update_layout(template='plotly_white', height=720, width=950, title='PnL comparison')
fig.update_xaxes(title='Date', row=2, col=1)
fig.update_yaxes(title='Cumulative PnL', row=1, col=1)
fig.update_yaxes(title='Daily PnL', row=2, col=1)
fig.show()

## Drawdown comparison

In [ ]:
drawdown_compare = nav_compare.divide(nav_compare.cummax()).sub(1.0)
fig = go.Figure()
fig.add_trace(go.Scatter(x=drawdown_compare.index, y=drawdown_compare['baseline'], mode='lines', name='Baseline'))
fig.add_trace(go.Scatter(x=drawdown_compare.index, y=drawdown_compare['dynamic'], mode='lines', name='Dynamic'))
fig.update_layout(template='plotly_white', height=420, width=950, title='Drawdown comparison')
fig.update_xaxes(title='Date')
fig.update_yaxes(title='Drawdown')
fig.show()

## Signal vs future realized minus IV

In [ ]:
future_horizon = 21
future_realized_vol = (spot.sort_index().pipe(lambda s: s.astype(float).pipe(__import__('numpy').log)).diff().pow(2).shift(-1).rolling(future_horizon).mean() * 252).pow(0.5)
future_realized_vol = future_realized_vol.rename('future_realized_vol').reset_index().rename(columns={'index': 'date'})
signal_future = signal.merge(future_realized_vol, on='date', how='left').dropna()
signal_future['future_realized_minus_iv'] = signal_future['future_realized_vol'] - signal_future['iv_reference']
fig = go.Figure()
fig.add_trace(go.Scatter(x=signal_future['vol_signal'], y=signal_future['future_realized_minus_iv'], mode='markers', name='Observations', marker=dict(size=7, opacity=0.6)))
fig.update_layout(template='plotly_white', height=420, width=900, title='Signal vs future realized volatility minus IV')
fig.update_xaxes(title='vol_signal')
fig.update_yaxes(title=f'future {future_horizon}d realized vol - IV')
fig.show()
signal_future[['vol_signal', 'future_realized_minus_iv']].corr()

## Bucket analysis

In [ ]:
bucket_df = signal_future.copy()
bucket_df['signal_bucket'] = pd.qcut(bucket_df['vol_signal'], q=5, labels=False, duplicates='drop')
bucket_summary = bucket_df.groupby('signal_bucket')[['future_realized_minus_iv', 'allocation_multiplier']].mean().reset_index()
fig = make_subplots(rows=1, cols=2, subplot_titles=('Future realized vol - IV by signal bucket', 'Allocation by signal bucket'))
fig.add_trace(go.Bar(x=bucket_summary['signal_bucket'], y=bucket_summary['future_realized_minus_iv'], name='Future realized - IV'), row=1, col=1)
fig.add_trace(go.Bar(x=bucket_summary['signal_bucket'], y=bucket_summary['allocation_multiplier'], name='Allocation'), row=1, col=2)
fig.update_layout(template='plotly_white', height=420, width=950, title='Bucket analysis')
fig.show()
bucket_summary

## Exposure vs volatility

In [ ]:
exposure_vol = signal.merge(sigma_hat[['date', 'forecast_sigma_hat']], on='date', how='left')
fig = go.Figure()
fig.add_trace(go.Scatter(x=exposure_vol['forecast_sigma_hat'], y=exposure_vol['allocation_multiplier'], mode='markers', name='Observations', marker=dict(size=7, opacity=0.6)))
fig.update_layout(template='plotly_white', height=420, width=900, title='Exposure vs forecast volatility')
fig.update_xaxes(title='forecast_sigma_hat')
fig.update_yaxes(title='allocation_multiplier')
fig.show()